In [1]:
import os
import sys
from pathlib import Path

NOTEBOOK_ROOT = Path("/scratch/jq2uw/MME/instruct_vlm_edit")
os.chdir(NOTEBOOK_ROOT)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.append(str(NOTEBOOK_ROOT))
os.chdir(NOTEBOOK_ROOT)

from revlm import *
import argparse
args = argparse.Namespace(split="all", dataset_name="aokvqa")
config = configure_args(args, config_path=None)
ds = VQADataset(configure_args(args, config_path=None))
aokvqa_df = ds.load_df()

args = argparse.Namespace(split="all", dataset_name="fvqa")
config = configure_args(args, config_path=None)
ds = VQADataset(config)
fvqa_df = ds.load_df()

Task evaluation metrics will be saved to results/te/ft/llava-1.5-7b-hf/aokvqa
Edit evaluation metrics will be saved to results/ee/ft/llava-1.5-7b-hf/aokvqa
Predictions will be saved to results/pred/llava-1.5-7b-hf/aokvqa
Unified filename to save: mc_all.json
Task evaluation metrics will be saved to results/te/ft/llava-1.5-7b-hf/aokvqa
Edit evaluation metrics will be saved to results/ee/ft/llava-1.5-7b-hf/aokvqa
Predictions will be saved to results/pred/llava-1.5-7b-hf/aokvqa
Unified filename to save: mc_all.json
Task evaluation metrics will be saved to results/te/ft/llava-1.5-7b-hf/fvqa
Edit evaluation metrics will be saved to results/ee/ft/llava-1.5-7b-hf/fvqa
Predictions will be saved to results/pred/llava-1.5-7b-hf/fvqa
Unified filename to save: mc_all.json


In [2]:
from huggingface_hub import snapshot_download
import pandas as pd
# Download caption parquet files from HF
repo_id = "JJoy333/RationaleVQA"
local_root = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=["i_gen/*.parquet"],
)
# Load caption dataframes
fvqa_df_caption = pd.read_parquet(os.path.join(local_root, "i_gen", "fvqa.parquet"))
aokvqa_df_caption = pd.read_parquet(os.path.join(local_root, "i_gen", "aokvqa.parquet"))
print(f"Loaded FVQA captions: {len(fvqa_df_caption)} rows")
print(f"Loaded AOKVQA captions: {len(aokvqa_df_caption)} rows")
print(f"\nFVQA caption columns: {fvqa_df_caption.columns.tolist()}")
print(f"\nAOKVQA caption columns: {aokvqa_df_caption.columns.tolist()}")



Loaded FVQA captions: 2190 rows
Loaded AOKVQA captions: 17656 rows

FVQA caption columns: ['image_info_source', 'image_info_id', 'image_path', 'caption']

AOKVQA caption columns: ['image_info_source', 'image_info_id', 'image_path', 'caption']


In [ ]:
os.chdir(NOTEBOOK_ROOT)
from revlm.metrics.utils.image_generality import ImageGenerator
from data_raw.tokens import HF_TOKEN

model_name = "sd3"

# Take first row from FVQA captions
dataset_name = "fvqa"
first_row = fvqa_df_caption.iloc[0]
caption = first_row['caption']
image_id = first_row['image_info_id']
print(f"Generating image for caption: {caption}")
print(f"Image ID: {image_id}\n")

# Generate image
output_dir = Path("data/related_image") / dataset_name / image_id
output_dir.mkdir(parents=True, exist_ok=True)

generator = ImageGenerator(model_name, token=HF_TOKEN)
for i in range(5):
    save_fname = output_dir / f"{model_name}_{i}.png"
    image = generator.generate(caption, save_path=str(save_fname))
    print(f"Saved to {save_fname}")
print("Done!")


Generating image for caption: The red, double decker bus is driving past other buses. 
Image ID: val_1584



RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-69191bed-2204cdb844660a5269e2d3fb;a0d171de-7076-4ecc-af12-f9567e7e9b7f)

Repository Not Found for url: https://huggingface.co/api/models/stabilityai/stable-diffusion-2-1.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication